In [0]:
#creating the execution log table and helper function
import uuid
from datetime import datetime
from pyspark.sql import functions as F

# 1. Create Audit Execution Log Table
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.tmdb.pipeline_execution_logs (
    log_id STRING,
    layer_processed STRING,
    load_type STRING,
    batch_id STRING,
    execution_start_time TIMESTAMP,
    execution_end_time TIMESTAMP,
    status STRING,
    records_processed LONG,
    error_message STRING
)
USING DELTA
""")

# 2. Logging function
def log_pipeline_execution(layer, load_type, batch_id, start_time, end_time, status, records, error_msg="NONE"):
    log_data = [(
        str(uuid.uuid4()),
        layer,
        load_type,
        batch_id,
        start_time,
        end_time,
        status,
        records,
        error_msg
    )]
    schema = spark.table("workspace.tmdb.pipeline_execution_logs").schema
    log_df = spark.createDataFrame(log_data, schema=schema)
    log_df.write.format("delta").mode("append").saveAsTable("workspace.tmdb.pipeline_execution_logs")
    print(f"Logged execution for layer '{layer}': Status={status}, Records={records}")